# USN Base (350M) Pretraining on Kaggle (2x T4)

This notebook pretrains a USN-Base model (350M parameters) on:
1. English general text (OpenWebText subset)
2. Math dataset (GSM8K style)

**Hardware**: 2x NVIDIA T4 GPUs (Kaggle accelerator)
**Target**: ~2 hours training time
**Tokenizer**: GPT-2 (50257 vocab)
**Architecture**: USN (Unified State Network) — no attention, O(n) training, O(1) inference

## 1. Setup & Installation

In [ ]:
!pip install -q usn transformers datasets tokenizers accelerate
!pip install -q torch --index-url https://download.pytorch.org/whl/cu118

In [ ]:
import os
import time
import math
import logging

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler
from torch.nn.parallel import DataParallel

from transformers import GPT2Tokenizer
from datasets import load_dataset

import usn
from usn import USNConfig, USNModel, USNTrainingConfig
from usn.losses.cross_entropy import USNCrossEntropyLoss
from usn.optim.factory import OptimizerFactory
from usn.optim.schedulers import WarmupCosineScheduler
from usn.serialization.writer import USNWriter

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s')
logger = logging.getLogger(__name__)

print(f'USN version: {usn.__version__}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')